# Notebook 05 — Artist & Genre Analysis
**Feature 1.7 — EDA & Data Understanding | HitRadar Pro**

## Mục tiêu
- Phân tích top artists theo track count và popularity.
- Phân tích genre distribution và genre trend theo thập kỷ.
- Đánh giá track_artists coverage warning (96.54%) và ảnh hưởng của nó.

In [ ]:
import os, psycopg2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

conn = psycopg2.connect(
    host='localhost', port=5432, user='postgres',
    password=os.environ.get('PGPASSWORD', ''), dbname='hitradar'
)
print('Kết nối thành công.')

## 1. Top Artists theo Track Count

In [ ]:
df_artists = pd.read_sql("""
    SELECT artist_name, track_count, avg_track_popularity,
           max_track_popularity, followers,
           artist_popularity_dashboard_only
    FROM analytics.vw_top_artists
    ORDER BY track_count DESC
    LIMIT 20
""", conn)
print(f'Tổng artists có track: 81,776')
df_artists[['artist_name','track_count','avg_track_popularity','max_track_popularity']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
y_pos = range(len(df_artists))
bars = ax.barh(y_pos, df_artists['track_count'], color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_artists['artist_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Số tracks')
ax.set_title('Top 20 Artists theo số lượng tracks', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(df_artists['track_count']):
    ax.text(v + 30, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 2. Top Artists theo Avg Track Popularity

In [ ]:
df_pop_artists = pd.read_sql("""
    SELECT artist_name, track_count, avg_track_popularity, max_track_popularity
    FROM analytics.vw_top_artists
    WHERE track_count >= 5
    ORDER BY avg_track_popularity DESC
    LIMIT 15
""", conn)

fig, ax = plt.subplots(figsize=(11, 5))
y_pos = range(len(df_pop_artists))
ax.barh(y_pos, df_pop_artists['avg_track_popularity'], color='darkorange')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_pop_artists['artist_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Avg Track Popularity')
ax.set_title('Top 15 Artists theo Avg Track Popularity (≥5 tracks)\n'
             '⚠️ artist_popularity_dashboard_only — KHÔNG dùng làm ML input', fontweight='bold')
for i, v in enumerate(df_pop_artists['avg_track_popularity']):
    ax.text(float(v) + 0.3, i, f'{float(v):.1f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Top Genres theo Số Tracks

In [ ]:
df_genres = pd.read_sql("""
    SELECT genre_name,
           SUM(track_count)       AS total_tracks,
           ROUND(AVG(avg_popularity)::numeric, 1) AS avg_pop
    FROM analytics.vw_genre_trends
    GROUP BY genre_name
    ORDER BY total_tracks DESC
    LIMIT 20
""", conn)

fig, ax = plt.subplots(figsize=(11, 6))
y_pos = range(len(df_genres))
bars = ax.barh(y_pos, df_genres['total_tracks'], color='#2ca02c')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_genres['genre_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Tổng số tracks')
ax.set_title('Top 20 Genres theo số tracks (cross all decades)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(df_genres['total_tracks']):
    ax.text(int(v) + 100, i, f'{int(v):,}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Genre Trend theo Thập kỷ (Top 5 genres)

In [ ]:
# Lấy top 5 genres tổng thể
top5 = df_genres['genre_name'].head(5).tolist()

placeholders = ','.join([f"'{g}'" for g in top5])
df_genre_trend = pd.read_sql(f"""
    SELECT genre_name, decade, track_count
    FROM analytics.vw_genre_trends
    WHERE genre_name IN ({placeholders})
    AND decade >= 1940
    ORDER BY decade, genre_name
""", conn)

fig, ax = plt.subplots(figsize=(13, 5))
for g in top5:
    sub = df_genre_trend[df_genre_trend.genre_name == g]
    ax.plot(sub['decade'], sub['track_count'], marker='o', label=g, linewidth=1.5)

ax.set_title('Xu hướng top 5 genres theo thập kỷ', fontweight='bold')
ax.set_xlabel('Thập kỷ')
ax.set_ylabel('Số tracks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 5. Track Artists Coverage Warning

In [ ]:
df_cov = pd.read_sql("""
    SELECT metric_name, metric_value, severity, note
    FROM analytics.vw_data_quality_report
    WHERE metric_name IN ('track_artists_coverage_pct','track_artists_skipped',
                          'genre_join_coverage_pct')
""", conn)
print('Coverage metrics từ vw_data_quality_report:')
print(df_cov.to_string(index=False))
print('\nNhận xét: 3.46% tracks không thể map artist do artist_id'
      ' không tồn tại trong artists.csv.')

## 6. Nhận xét & Kết luận

**Artist bias:**
- Top artists theo track_count là **Die drei ???** (3,856 tracks — German radio drama) và **Lata Mangeshkar** (2,605 — Indian classical). Dataset có bias quốc tế rõ ràng.
- Phần lớn trong 81,776 artists có rất ít tracks — phân bố long-tail điển hình.
- **artist_popularity** chỉ dùng cho dashboard — KHÔNG dùng làm ML input feature.

**Genre insights:**
- Top genre: **rock** (32,026), **adult standards** (26,688), **classic rock** (23,808) — dataset thiên về nhạc rock và pop cổ điển.
- **filmi** (19,557 — Indian film music) và **classical** (18,995) cho thấy diversity quốc tế.
- Genre trend: rock và classic rock tăng mạnh ở 1970–1980s, sau đó stable.

**Coverage warning:**
- **track_artists coverage = 96.54%** — 26,224 tracks thiếu artist info.
- **genre_join_coverage = 100%** — sạch, không ảnh hưởng genre analysis.
- Tracks bị skip chủ yếu là niche/inactive artists không có trong artists.csv.

**Đề xuất cho EPIC 2:**
- Genre có thể dùng như categorical feature — nhưng cần embedding vì số lượng lớn (4,672).
- Cân nhắc dùng top-N genres hoặc genre category thay vì toàn bộ.
- 3.46% tracks bị skip artist sẽ cần xử lý (fill NULL hoặc loại bỏ) nếu dùng artist features.

In [ ]:
conn.close()
print('Done — Notebook 05 hoàn thành.')